# Fundamentals of Data Science - Network Science Lab

# Lab Session 07 - Network Analysis

In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "igraph": "igraph==1.0.0", "numpy": "numpy", "pandas": "pandas",
    "scipy": "scipy", "matplotlib": "matplotlib"
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import random
import time
import igraph as ig
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display

SEED = 2411
ig.set_random_number_generator(random.Random(SEED))
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False})
BLUE, ORANGE, GREEN = "#2864A5", "#DA7B27", "#238678"
pd.set_option("display.max_columns", 12)
print("igraph:", ig.__version__)

## Bipartite Networks

A bipartite network consists of two distinct types of vertices, with edges connecting only nodes belonging to different groups.

In [ ]:
# Keep track of whether some people visited a certain place or not this week.
people = ["Edoardo", "Matteo", "Irene", "Simon", "Ciro"]
places = ["Bar", "Library", "Office", "Cinema"]

B_toy = np.array([
    [1, 0, 0, 1],  # Edoardo.
    [1, 1, 1, 0],  # Matteo.
    [0, 1, 1, 0],  # Irene.
    [1, 0, 1, 0],  # Simon.
    [0, 0, 1, 0],  # Ciro.
], dtype=np.int64)

display(pd.DataFrame(B_toy, index=people, columns=places))

# Use the biadjacency matrix to create a bipartite network.
toy = ig.Graph.Biadjacency(B_toy.tolist(), directed=False)
toy.vs["name"] = people + places # Assign names to each node.
print(toy.summary())
print("Vertex types:", toy.vs["type"])
print(toy.is_bipartite()) # Directly check whether a network is bipartite or not.
print(all(toy.vs[e.source]["type"] != toy.vs[e.target]["type"] for e in toy.es)) # The type attribute is a Boolean that distinguishes between the two groups.
# print(toy.summary())

# Plot the bipartite network.
nu, ns = B_toy.shape # For the plot layout.
layout = [(0, y) for y in np.linspace(0, 1, nu)]
layout += [(1, y) for y in np.linspace(0, 1, ns)]
fig, ax = plt.subplots(figsize=(9, 5))
ig.plot(toy, target=ax, layout=layout,
        vertex_color=[BLUE] * nu + [ORANGE] * ns,
        vertex_shape=["circle"] * nu + ["rectangle"] * ns,
        vertex_size=24, vertex_label=toy.vs["name"], vertex_label_size=10,
        vertex_label_dist=1.5, edge_color="#A5ADB5", edge_width=1.6)
ax.set_title("People on the Left / Places on the Right")
plt.show()

One-mode projection allows to create co-occurrence networks where only one type of node appears.

For a bipartite network $G = (V, E)$, where $V = V_i \cup V_j$ with $V_i \cap V_j = \emptyset$, this procedure can be performed by computing the matrix $A_i = II^T$, which represents the projection onto partition $V_i$, or the matrix $A_j = I^TI$, which represents the projection onto partition $V_j$.

In [ ]:
# Perform one-mode projection onto both partitions.
C_people = B_toy @ B_toy.T
C_places = B_toy.T @ B_toy

# print("People projection, including the diagonal:")
# display(pd.DataFrame(C_people, index=people, columns=people))
# print("The diagonal is the bipartite user degree:", np.diag(C_people))

# Remove eventual self-loops from the co-occurrence networks.
W_people = C_people.copy()
W_places = C_places.copy()
np.fill_diagonal(W_people, 0)
np.fill_diagonal(W_places, 0)
A_people = (W_people > 0).astype(int)

# Analyse the summary of each projection.
print("Weighted people projection:")
display(pd.DataFrame(W_people, index=people, columns=people))
print("Weighted places projection:")
display(pd.DataFrame(W_places, index=places, columns=places))

While bipartite networks are often summarized using a biadjacency matrix $I$, it is also possible to construct a full adjacency matrix as follows:

$$A = \begin{pmatrix}
  0 & I \\
  I^T & 0 \\
\end{pmatrix}$$

In [ ]:
# Get the full adjacency matrix from the bipartite graph.
A_bip = np.array(toy.get_adjacency().data)
labels = toy.vs["name"]
display(pd.DataFrame(A_bip, index=labels, columns=labels))

# Analyse interactions between nodes in the same partition by squaring the full adjacency matrix.
A_bip_squared = A_bip @ A_bip
display(pd.DataFrame(A_bip_squared, index=labels, columns=labels)) # Note that the diagonal entries are typically set to zero.

Overall, the main issue of bipartite networks lies in the fact that, even for sparse networks, the resulting output can be very dense and expensive to storage.

Since sparse matrices cannot make the resulting projection small, it is common practice to filter the network before making a projection in order to reduce computational costs.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load the dataset.
reddit_data = pd.read_csv("subreddit_mh_random_sample_25k.csv")
display(reddit_data.head())

# Look at the interactions in the dataset.
interactions = (reddit_data.groupby(["username", "subreddit"], as_index=False)["weight"]
               .sum().sort_values(["username", "subreddit"]).reset_index(drop=True))
print(f"{interactions.username.nunique():,} users; {interactions.subreddit.nunique():,} subreddits")
print(f"{len(interactions):,} distinct participation links; {interactions.weight.sum():,} interactions (posts or comments)")

# Group data into user–subreddit pairs.
interactions = (
    reddit_data.groupby(["username", "subreddit"], as_index=False)["weight"]
    .sum()
)
# For each subreddit, store the number of users and total posts/comments.
subreddit_summary = interactions.groupby("subreddit").agg(
    n_users=("username", "nunique"),
    n_interactions=("weight", "sum")
)
print("\nTop 10 subreddits by number of users:")
display(subreddit_summary.sort_values("n_users", ascending=False).head(10))

# For each user, store the number of subreddits and total posts/comments.
user_summary = interactions.groupby("username").agg(
    n_subreddits=("subreddit", "nunique"),
    n_interactions=("weight", "sum")
)
n_multi = (user_summary["n_subreddits"] > 1).sum()
print(f"Users active in multiple subreddits: {n_multi:,}")
print(f"Share of users: {n_multi / len(user_summary):.1%}")

In [ ]:
# Create a bipartite graph from username, category and weight.
g_bip = ig.Graph.TupleList(
    reddit_data[["username", "subreddit", "weight"]].itertuples(index=False, name=None),
    directed=False,
    edge_attrs=["weight"]
)

# Group nodes by considering True = Subreddit and False = User.
categories = set(reddit_data["subreddit"])
g_bip.vs["type"] = [name in categories for name in g_bip.vs["name"]]

# Look at a summary of the network and check whether it is bipartite or not.
print(g_bip.summary())
print(g_bip.is_bipartite())

In [ ]:
# Select users connected to at least two subreddits.
users = g_bip.vs.select(type_eq=False)
multi_users = [v for v in users if g_bip.degree(v.index) >= 2]

# Take the first 10 such users and fine the subreddits connected to those users.
selected_users = multi_users[:5]
subreddits = set()
for user in selected_users:
    subreddits.update(g_bip.neighbors(user.index))

# Build a subgraph containing the selected users and their subreddits.
vertices = [user.index for user in selected_users] + list(subreddits)
g_sub = g_bip.induced_subgraph(vertices)
print(g_sub.summary())

# Plot the graph, showing users as blue nodes and subreddits as orange nodes.
fig, ax = plt.subplots(figsize=(18, 9))

graph_plot = ig.plot(
    g_sub,
    target=ax,
    layout=g_sub.layout_bipartite(),
    vertex_color=["orange" if t else "lightblue" for t in g_sub.vs["type"]],
    vertex_size=20,
    vertex_label=g_sub.vs["name"],
    vertex_label_size=8,
    vertex_label_dist=2,
    edge_color="gray",
    edge_width=1
)

fig.canvas.draw()
for item in graph_plot.get_children():
    if isinstance(item, plt.Text):
        item.set_rotation(45)

plt.show()

In [ ]:
# Select each set of vertices and get their degrees.
users = g_bip.vs.select(type_eq=False)
subreddits = g_bip.vs.select(type_eq=True)
user_degrees = g_bip.degree(users.indices)
subreddit_degrees = g_bip.degree(subreddits.indices)

# Plot the degree distribution for each partition.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Users: Discrete degree values.
values, counts = np.unique(user_degrees, return_counts=True)
axes[0].bar(values, counts, color="lightblue", edgecolor="black")
axes[0].set_xlabel("Number of Subreddits")
axes[0].set_ylabel("Number of Users")
axes[0].set_title("User Degree Distribution")

# Subreddits: Audience sizes.
axes[1].hist(subreddit_degrees, bins=20, color="orange", edgecolor="black")
axes[1].set_xlabel("Number of Users")
axes[1].set_ylabel("Number of Subreddits")
axes[1].set_title("Subreddit Degree Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Separate the two types of vertices.
user_vertices = g_bip.vs.select(type_eq=False)
sub_vertices = g_bip.vs.select(type_eq=True)

# Compute user-level statistics.
user_stats = pd.DataFrame({
    "username": user_vertices["name"],
    "n_subreddits": g_bip.degree(user_vertices.indices),
    "n_interactions": g_bip.strength(user_vertices.indices, weights="weight")
})
print("\nUsers active in the most subreddits:")
display(user_stats.sort_values("n_subreddits", ascending=False).head(8))

# Compute subreddit-level statistics.
sub_stats = pd.DataFrame({
    "subreddit": sub_vertices["name"],
    "n_users": g_bip.degree(sub_vertices.indices),
    "n_interactions": g_bip.strength(sub_vertices.indices, weights="weight")
})
print("\nSubreddits with the most users:")
display(sub_stats.sort_values("n_users", ascending=False))

In [ ]:
# Study the density of the network by considering the fraction of possible user–subreddit edges that exist.
bipartite_density = g_bip.ecount() / (len(user_vertices) * len(sub_vertices))

print(f"Users: {len(user_vertices):,}")
print(f"Subreddits: {len(sub_vertices):,}")
print(f"Edges: {g_bip.ecount():,}")
print(f"Bipartite density: {bipartite_density:.3f}")

In [ ]:
# Filter the network by taking only the users that satisfy the requirement on the minimum number of interactions.
MIN_USER_INTERACTIONS = 10

user_activity = reddit_data.groupby("username")["weight"].sum()
eligible_users = user_activity[
    user_activity >= MIN_USER_INTERACTIONS
].index
filtered = reddit_data[
    reddit_data["username"].isin(eligible_users)
].copy()

# Analyse the results of the network.
print(f"Users before: {reddit_data['username'].nunique():,}")
print(f"Edges before:  {reddit_data.shape[0]:,}")
print(f"Users after:  {len(eligible_users):,}")
print(f"Edges after:  {len(filtered):,}")

In [ ]:
# Filter the network by taking only the users that satisfy the requirement on the minimum number of subreddits.
MIN_SUBREDDITS = 3

user_subreddits = reddit_data.groupby("username")["subreddit"].nunique()
active_users = user_subreddits[
    user_subreddits >= MIN_SUBREDDITS
].index
filtered = reddit_data[
    reddit_data["username"].isin(active_users)
].copy()

print(f"Users before: {reddit_data['username'].nunique():,}")
print(f"Edges before: {len(reddit_data):,}")
print(f"Users after:  {filtered['username'].nunique():,}")
print(f"Edges after:  {len(filtered):,}")

In [ ]:
# Project the network onto the user group.
g_users = g_bip.bipartite_projection(
    types="type", multiplicity=True, which = 0
)

print(g_users.summary())
print(f"Users: {g_users.vcount():,}")
print(f"Edges: {g_users.ecount():,}")
print(f"Density: {g_users.density():.4f}")

# Filter the co-occurrence network by taking only the users that satisfy the requirement on the minimum number of shared subreddits.
MIN_SHARED_SUBREDDITS = 2
g_threshold = g_users.copy()
g_threshold.delete_edges(g_threshold.es.select(weight_lt=MIN_SHARED_SUBREDDITS))
print(f"At least {MIN_SHARED_SUBREDDITS} shared subreddits: {g_threshold.ecount():,} edges")
print("Vertices retained:", g_threshold.vcount())
print(f"Density: {g_threshold.density():.4f}")

# Keep only vertices with degree > 0.
g_threshold_active = g_threshold.induced_subgraph(
    [v.index for v in g_threshold.vs if v.degree() > 0]
)
print(g_threshold_active.summary())
print(f"Density without isolated nodes: {g_threshold_active.density():.4f}")

In [ ]:
def disparity_filter(graph, alpha=0.05):
  # The disparity filter keeps an edge if it is significant for at least one endpoint.

  # Check whether it is possible to apply the disparity filter for the input data.
  if not 0 < alpha < 1:
    raise ValueError("alpha must be between 0 and 1.")

  if graph.is_directed() or not graph.is_simple():
    raise ValueError("The graph must be simple and undirected.")

  if graph.ecount() == 0:
    return graph.copy()

  weights = np.asarray(graph.es["weight"], dtype=float)

  if not np.all(np.isfinite(weights) & (weights > 0)):
    raise ValueError("All edge weights must be positive and finite.")

  # Extract information about vertices and edges.
  degrees = np.asarray(graph.degree())
  strengths = np.asarray(graph.strength(weights="weight"))
  edges = np.asarray(graph.get_edgelist(), dtype=int)

  # Decide which edges to keep by checking each edge from both endpoints.
  keep = np.zeros(graph.ecount(), dtype=bool)
  for side in (0, 1):
    nodes = edges[:, side]

    # The local test is defined for nodes with degree > 1.
    valid = degrees[nodes] > 1
    shares = weights[valid] / strengths[nodes[valid]]
    p_values = (1 - np.clip(shares, 0, 1)) ** (
        degrees[nodes[valid]] - 1
    )

    keep[valid] |= p_values < alpha

  # Isolated pairs of degree-one nodes are normally retained becauase neither endpoint has other edges for local comparisons.
  isolated_pairs = (
      (degrees[edges[:, 0]] == 1) &
      (degrees[edges[:, 1]] == 1)
  )
  keep |= isolated_pairs

  # Return a filtered copy of the input network.
  backbone = graph.copy()
  backbone.delete_edges(np.flatnonzero(~keep).tolist())

  return backbone

In [ ]:
# Apply the disparity filter on a network.
g_disparity = disparity_filter(g_users, alpha=0.20)
g_disparity = g_disparity.induced_subgraph(
    [v.index for v in g_disparity.vs if v.degree() > 0]) # Remove eventual isolated nodes.

# Make a comparison between the original network and the filtered network.
print("Before disparity filter:")
print(g_users.summary())

print("\nAfter disparity filter:")
print(g_disparity.summary())

print(f"\nEdges retained: {g_disparity.ecount():,}")
print(f"\nNodes retained: {g_disparity.vcount():,}")

In [ ]:
# Analyse the effects of the disparity filter for various values of alpha.
alphas = [0.01, 0.05, 0.10, 0.20]
results = []

for alpha in alphas:
  start = time.perf_counter()

  # Apply the disparity filter to the original projection.
  backbone = disparity_filter(g_users, alpha=alpha)
  degrees = np.asarray(backbone.degree())
  # n_isolates = np.sum(degrees == 0)
  backbone = backbone.induced_subgraph(
  [v.index for v in backbone.vs if v.degree() > 0]) # Remove eventual isolated nodes.

  results.append({
      "alpha": alpha,
      "users": backbone.vcount(),
      "edges": backbone.ecount(),
      "edges_retained_pct": 100 * backbone.ecount() / g_users.ecount(),
      # "isolates": n_isolates,
      # "active_users": np.sum(degrees > 0),
      "density": backbone.density(),
      "seconds": time.perf_counter() - start
  })

  print(f"Finished alpha = {alpha:.2f}")

  # Release this backbone before computing the next one.
  del backbone

disparity_summary = pd.DataFrame(results)

display(disparity_summary.round(4))

In [ ]:
# Analyse the edge weight distribution of the filtered network.
weights = np.asarray(g_users.es["weight"])
values, counts = np.unique(weights, return_counts=True)

weight_distribution = pd.DataFrame({
    "shared_subreddits": values,
    "n_edges": counts
})

display(weight_distribution)

# Plot the edge weight distribution,
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(values, counts, color="steelblue", edgecolor="black")
ax.set_yscale("log")
ax.set_xlabel("Number of Shared Subreddits")
ax.set_ylabel("Number of Edges")
ax.set_title("Distribution of Projected Edge Weights")

plt.tight_layout()
plt.show()

# Free the large temporary array.
del weights

In [ ]:
# Analyse the user projection obtained with alpha = 0.2.
g = g_disparity
print(g.summary())
print(f"Users: {g.vcount():,}")
print(f"Edges: {g.ecount():,}")
print(f"Density: {g.density():.4f}")

# Look at the connected components in the network.
components = g.connected_components()
print(f"\nConnected components: {len(components):,}")
print(f"Isolates: {sum(d == 0 for d in g.degree()):,}")

# Analyse the largest connected component of the network.
g_lcc = components.giant()
print(f"Largest component: {g_lcc.vcount():,} users")
print(f"Edges in largest component: {g_lcc.ecount():,}")
print(f"Share of users in largest component: "
      f"{g_lcc.vcount() / g.vcount():.1%}")

In [ ]:
# Study the degree distribution of the network.
degrees = np.asarray(g_lcc.degree())
print(f"Mean degree: {degrees.mean():.2f}")
print(f"Median degree: {np.median(degrees):.0f}")
print(f"Maximum degree: {degrees.max():}")

# Plot the CCDF of the distribution.
values, counts = np.unique(degrees, return_counts=True)
ccdf = counts[::-1].cumsum()[::-1] / len(degrees)

fig, ax = plt.subplots(figsize=(7, 5))

ax.step(values, ccdf, where="post", color="steelblue")
ax.set_xlabel("Degree: Number of Connected Users")
ax.set_ylabel("P(Degree ≥ k)")
ax.set_title("User Degree Distribution")

plt.tight_layout()
plt.show()

In [ ]:
degree_df = pd.DataFrame({
    "username": g_lcc.vs["name"],
    "degree": degrees
})
display(degree_df.sort_values("degree", ascending=False).head(10))

# Find the top 10 users by degree in the filtered network.
top_users = (
    degree_df.sort_values("degree", ascending=False)
    .head(10)["username"]
)

# Analyse the activity of these users.
top_user_activity = (
    reddit_data[reddit_data["username"].isin(top_users)]
    .sort_values(["username", "weight"], ascending=[True, False])
)

display(top_user_activity)

## Network Descriptives

The local clustering coefficient measures how connected a user's
neighbors are to one another, meaning that

$$c_i = \begin{cases}
  \frac{2m_i}{k_i(k_i - 1)} &\quad\text{if $k_i > 1$} \\
  0 &\quad\text{if $k_i \leq 1$} \\
\end{cases}$$

In a projected network, high clustering should be interpreted carefully because three users who participate in the same subreddit form a
triangle in the ordinary projection, even if they never interact directly.

In [ ]:
# Analyse local clustering for each user.
local_clustering = np.asarray(
    g_lcc.transitivity_local_undirected(mode="nan")
)
valid_clustering = local_clustering[np.isfinite(local_clustering)] # Exclude users with undefined values ord egree < 2.
print(f"Users with defined clustering: {len(valid_clustering):,}")
print(f"Average local clustering: {valid_clustering.mean():.3f}")
print(f"Median local clustering: {np.median(valid_clustering):.3f}")

# Plot the clustering coefficient distribution.
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(valid_clustering, bins=20, color="mediumseagreen",
        edgecolor="black")
ax.set_xlabel("Local Clustering Coefficient")
ax.set_ylabel("Number of Users")
ax.set_title("Clustering in the User Projection")

plt.tight_layout()
plt.show()

Closeness centrality measures how short a user's paths to other users are on average, meaning that a user with high closeness can reach other users through relatively few steps in the network.

In a projected network, a path is a chain of shared-participation links, meaning that high closeness describes a user's position in the projected network.

In [ ]:
# Analyse closeness centrality for each user.
closeness = np.asarray(
    g_lcc.closeness(weights=None, normalized=True)
)
closeness_df = pd.DataFrame({
    "username": g_lcc.vs["name"],
    "closeness": closeness
})
print("Highest closeness:")
display(closeness_df.sort_values("closeness", ascending=False).head(10))

# Plot the closeness centrality distribution.
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(closeness, bins=30, color="darkorange", edgecolor="black")
ax.set_xlabel("Closeness Centrality")
ax.set_ylabel("Number of Users")
ax.set_title("Closeness in the Largest Connected Component")

plt.tight_layout()
plt.show()

Betweenness centrality measures how often a user lies on shortest paths
between other pairs of users, meaning that a user with high betweenness may occupy a connecting position between different parts of the network.

Since exact betweenness can be computationally expensive and depends on the number of possible node pairs in a network, it can be interpreted by normalizing it over the maximum possible number of other users, resulting in

$$Betweenness_{\text{norm}}(v) = \frac{2Betweenness(v)}{(n - 1)(n - 2)}$$

In [ ]:
# Analyse betweenness centrality within the largest connected component.
betweenness = np.asarray(
    g_lcc.betweenness(directed=False, weights=None)
)
betweenness_df = pd.DataFrame({
    "username": g_lcc.vs["name"],
    "betweenness": betweenness
})

print("Highest betweenness:")
display(
    betweenness_df.sort_values("betweenness", ascending=False).head(10)
)

# Plot the betweenness centrality distribution.
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(betweenness, bins=30, color="mediumpurple", edgecolor="black")
ax.set_xlabel("Betweenness Centrality")
ax.set_ylabel("Number of Users")
ax.set_title("Betweenness in the Largest Connected Component")

plt.tight_layout()
plt.show()

In [ ]:
# Normalize betweenness centrality for the network.
n = g_lcc.vcount()
betweenness_df["betweenness_normalized"] = (
    betweenness_df["betweenness"] / ((n - 1) * (n - 2) / 2)
)
display(
    betweenness_df.sort_values("betweenness", ascending=False).head(10)
)

## Random Networks

The Erdős-Rényi model can be defined using two possible ensembles:
1. The $G(n, m)$ ensemble, which samples $m$ distinct undirected edges from all $\binom{n}{2}$ possible edges.
2. The $G(n, p)$ ensemble, which includes each possible edge independently with probability $p$.

In [ ]:
ig.set_random_number_generator(random.Random(2411)) # For reproducibility.

# Generate an Erdős–Rényi graph with the same n and m as the observed largest connected component from the projection.
g_er = ig.Graph.Erdos_Renyi(
    n=g_lcc.vcount(),
    m=g_lcc.ecount(),
    directed=False,
    loops=False
)
print("Observed network:")
print(g_lcc.summary())
print("\nErdős–Rényi network:")
print(g_er.summary())

# Analyse the connected components in the network.
er_components = g_er.connected_components()
g_er_lcc = er_components.giant()

print(f"ER components: {len(er_components):,}")
print(f"ER largest component: {g_er_lcc.vcount():,} users")
print(f"Share of ER nodes in largest component: "
      f"{g_er_lcc.vcount() / g_er.vcount():.1%}")

# Compare the Erdős–Rényi graph to the observed largest connected component.
comparison = pd.DataFrame({
    "Metric": [
        "Nodes",
        "Edges",
        "Density",
        "Mean degree",
        "Maximum degree",
        "Connected components",
        "Largest component size",
        "Average local clustering",
        # "Global transitivity"
    ],
    "Observed": [
        g_lcc.vcount(),
        g_lcc.ecount(),
        g_lcc.density(),
        np.mean(g_lcc.degree()),
        max(g_lcc.degree()),
        len(g_lcc.connected_components()),
        g_lcc.vcount(),
        g_lcc.transitivity_avglocal_undirected(mode="nan"),
        # g_lcc.transitivity_undirected(mode="nan")
    ],
    "ER": [
        g_er.vcount(),
        g_er.ecount(),
        g_er.density(),
        np.mean(g_er.degree()),
        max(g_er.degree()),
        len(er_components),
        g_er_lcc.vcount(),
        g_er.transitivity_avglocal_undirected(mode="nan"),
        # g_er.transitivity_undirected(mode="nan")
    ]
})
display(comparison.set_index("Metric").round(4))

# Plot the CCDF of the degree distribution of the desired networks.
def plot_ccdf(values, ax, label, color):
  # Plot P(X >= x) for a set of input values.
  values = np.asarray(values)
  x, counts = np.unique(values, return_counts=True)
  ccdf = counts[::-1].cumsum()[::-1] / counts.sum()

  ax.step(x, ccdf, where="post", label=label, color=color)

# Compare the empirical CCDF to the CCDF of the observed largest connected component.
fig, ax = plt.subplots(figsize=(7, 5))

plot_ccdf(g_lcc.degree(), ax, "Observed", "steelblue")
plot_ccdf(g_er.degree(), ax, "ER", "darkorange")

ax.set_xlabel("Degree")
ax.set_ylabel("P(Degree ≥ k)")
ax.set_title("Degree Distribution")
ax.legend()

plt.tight_layout()
plt.show()

Overall, the resulting Erdős-Rényi network has the same number of nodes and edges as the observed largest connected component, meaning that its density and mean match those of the observed network, although other properties of the starting graph are not preserved.

However, since a statistic changes from one Erdős-Rényi draw to another, it is necessary to analyse the overall variability among a large number of independent samples.

In [ ]:
# Start from the observed largest connected component.
n = g_lcc.vcount()
m = g_lcc.ecount()

N_RANDOM = 100

# Set igraph's random number generator.
ig.set_random_number_generator(random.Random(2411))

# Store network statistics and sorted degree values.
er_stats = []
er_degree_distributions = []

start = time.perf_counter()

for i in range(N_RANDOM):

  # Generate the G(n, m) ensemble.
  g_random = ig.Graph.Erdos_Renyi(
      n=n,
      m=m,
      directed=False,
      loops=False
  )

  random_degrees = np.asarray(g_random.degree())
  components = g_random.connected_components()

  er_stats.append({
      "mean_degree": random_degrees.mean(),
      "max_degree": random_degrees.max(),
      "degree_sd": random_degrees.std(),
      "clustering": g_random.transitivity_avglocal_undirected(mode="nan"),
      # "transitivity": g_random.transitivity_undirected(mode="nan"),
      "n_components": len(components),
      "lcc_size": max(components.sizes())
  })

  # Sort degrees within this graph for the distribution comparison.
  er_degree_distributions.append(np.sort(random_degrees))

  if (i + 1) % 10 == 0:
    print(f"Completed {i + 1}/{N_RANDOM} graphs")

# Save the results into a Pandas dataframe.
er_stats = pd.DataFrame(er_stats)
er_degree_distributions = np.asarray(er_degree_distributions)

print(f"\nTotal time: {time.perf_counter() - start:.1f} seconds")

# Analyse the overall comparisons.
observed_degrees = np.asarray(g_lcc.degree())

observed_stats = pd.Series({
    "mean_degree": observed_degrees.mean(),
    "max_degree": observed_degrees.max(),
    "degree_sd": observed_degrees.std(),
    "clustering": g_lcc.transitivity_avglocal_undirected(mode="nan"),
    # "transitivity": g_lcc.transitivity_undirected(mode="nan"),
    "n_components": len(g_lcc.connected_components()),
    "lcc_size": g_lcc.vcount()
})

comparison = pd.DataFrame({
    "Observed": observed_stats,
    "ER mean": er_stats.mean(),
    "ER 2.5%": er_stats.quantile(0.025),
    "ER 97.5%": er_stats.quantile(0.975)
})
display(comparison.round(4))

# Plot the overall distributions.
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    er_stats["clustering"].dropna(),
    bins=20,
    color="lightblue",
    edgecolor="black",
    label="ER Graphs"
)

ax.axvline(
    observed_stats["clustering"],
    color="red",
    linewidth=2,
    label="Observed"
)

ax.set_xlabel("Average Local Clustering")
ax.set_ylabel("Number of ER Graphs")
ax.set_title("Observed Clustering vs. ER Ensemble")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Determine the degrees to evaluate, which go from 1 up to the largest observed or ER degree.
max_degree = max(observed_degrees.max(), er_degree_distributions.max())
degree_values = np.arange(1, max_degree + 1)

# Define the observed CCDF as P(Degree >= k).
observed_ccdf = np.array([
    np.mean(observed_degrees >= k)
    for k in degree_values
])

# Define the CCDF for each Erdős-Rényi graph.
er_ccdfs = np.array([
    [np.mean(degrees >= k) for k in degree_values]
    for degrees in er_degree_distributions
])

# Define and plot the median CCDF across the Erdős-Rényi graphs.
er_median = np.median(er_ccdfs, axis=0)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    degree_values, er_median,
    color="darkorange", label="ER median"
)

ax.plot(
    degree_values, observed_ccdf,
    color="steelblue", label="Observed"
)

ax.set_xlabel("Degree $k$")
ax.set_ylabel("P(Degree ≥ $k$)")
ax.set_title("Degree Distribution: Observed vs. ER")
ax.legend()

plt.tight_layout()
plt.show()